# 06 · Artigo × código: divergências encontradas na migração e seu efeito nos resultados

Ao portar o código MATLAB (`bbnet10`) que gerou os números publicados, algumas diferenças
entre o texto do artigo e o código foram identificadas (detalhes em `docs/paper_vs_code.md`).
O pacote reproduz o código por padrão e expõe interruptores para as formas "corrigidas".
Este notebook quantifica o efeito de cada uma delas.

| # | Item | Código (padrão, reproduz o artigo) | Forma descrita no artigo / física |
|---|------|------|------|
| 1 | Pressão de descarga $P_G$ | nó n+4 (interconexão das linhas de gás, −1495 m) | nó n+5 (plataforma, +20 m) |
| 2 | Orientação do anular | geometria escrita de baixo para cima (termo hidrostático de escoamento ascendente) | gás desce: −90° |
| 3 | Diâmetro equivalente do anular | $(D_o^2 + D_i^2)/(D_o + D_i)$ | $D_o - D_i$ (hidráulico) |
| 4 | Custo das linhas (McCoy) | $D \cdot a_7$ | $D^{a_7}$ (Eq. 33) |
| 5 | Expoente politrópico no compressor | 1,4 | 1,25 (Tabela 16) |
| 6 | Potência nas correlações de custo | hp | kW (Eqs. 30 e 32) |
| 7 | Geometria do poço | 4 in; 1710 + 338 m; rugosidade 0 nas linhas de gás | 5 in; 1500 + 300 m; 100 µm (Tabela 2) |

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
REF = ROOT / "data" / "reference"
FIG = ROOT / "docs" / "paper" / "figures"
pd.set_option("display.precision", 4)
pd.set_option("display.width", 140)
plt.rcParams.update({"figure.dpi": 100, "figure.figsize": (7, 5), "axes.grid": True, "grid.alpha": 0.3})

In [2]:
import json
from dataclasses import replace
from gaslift import case1_single_well, case2_multi_well, SimulationOptions
from gaslift.economics import CostModel, CompressorParameters, evaluate_economics, pipeline_capital_cost, compressor_capital_cost, turbine_capital_cost
matlab = json.load(open(REF / "matlab_optimization_results.json"))
sim_opts = SimulationOptions(compute_profiles=False)

## Itens 1–3 · Lado do gás: onde se lê a pressão de descarga e como o anular é modelado

O lado do gás não realimenta o lado da produção (as especificações são vazões), portanto
a pressão de fundo, a produção máxima e as Tabelas 14–15 não mudam. O que muda é a pressão
de descarga do compressor e tudo que dela deriva (potência, custos, lucro). Avaliamos o ponto
"Max" da Tabela 13 (9,598 kg/s de óleo, 1,812 kg/s de gás).

In [3]:
ref = matlab["case1_maxprod_pi1_220bar"]
q_oil, q_gas = float(np.ravel(ref["q_oil"])[0]), float(np.ravel(ref["q_gas"])[0])
variants = {
    "code (paper numbers)": dict(annulus_orientation="matlab", annulus_diameter_model="matlab", node="gas_junction"),
    "1: discharge at platform node": dict(annulus_orientation="matlab", annulus_diameter_model="matlab", node="gas_inlet"),
    "2: physical annulus orientation": dict(annulus_orientation="physical", annulus_diameter_model="matlab", node="gas_junction"),
    "3: hydraulic annulus diameter": dict(annulus_orientation="matlab", annulus_diameter_model="hydraulic", node="gas_junction"),
    "1+2+3 combined": dict(annulus_orientation="physical", annulus_diameter_model="hydraulic", node="gas_inlet"),
}
rows = {}
for name, v in variants.items():
    case = case1_single_well(q_oil=q_oil, q_gas=q_gas, annulus_orientation=v["annulus_orientation"], annulus_diameter_model=v["annulus_diameter_model"])
    sol = case.simulator(sim_opts).solve()
    ec = evaluate_economics(sol, case.wells, CostModel(include_fuel_gas_cost=True, discharge_pressure_node=v["node"]))
    rows[name] = {"Downhole pressure (bar)": ec.downhole_pressure[0], "Injection point pressure (bar)": sol.pressure_bar("well_1:injection_point"),
                  "Annulus top pressure (bar)": sol.pressure_bar("well_1:annulus_top"), "Gas junction, node 9 (bar)": sol.pressure_bar("well_1:gas_junction"),
                  "Platform gas inlet, node 10 (bar)": sol.pressure_bar("well_1:gas_inlet"), "Discharge pressure used (bar)": ec.discharge_pressure,
                  "Compressor power (kW)": ec.compressor_power, "Fuel gas cost (10⁶ USD/yr)": ec.fuel_gas_cost}
pd.DataFrame(rows)

,code (paper numbers),1: discharge at platform node,2: physical annulus orientation,3: hydraulic annulus diameter,1+2+3 combined
Downhole pressure (bar),210.4021,210.4021,210.4021,210.4021,210.4021
Injection point pressure (bar),181.3322,181.3322,181.3322,181.3322,181.3322
Annulus top pressure (bar),204.1740,204.1740,160.7568,204.2303,160.8252
"Gas junction, node 9 (bar)",279.6790,279.6790,248.2676,279.7218,248.3143
"Platform gas inlet, node 10 (bar)",261.9190,261.9190,233.6970,261.9576,233.7388
Discharge pressure used (bar),279.6790,261.9190,248.2676,279.7218,233.7388
Compressor power (kW),1108.0632,1086.3838,1068.7959,1108.1139,1049.0889
Fuel gas cost (10⁶ USD/yr),1.7657,1.7312,1.7031,1.7658,1.6717


Com a orientação física do anular, a pressão no topo do anular cai cerca de 2× a coluna de
gás de 1710 m (≈ 43 bar) e a pressão no nó 9 cai ≈ 31 bar; lendo a descarga na plataforma,
subtrai-se ainda a coluna da linha vertical de 1515 m (≈ 18 bar). Como o trabalho de compressão
depende fracamente da razão de pressões (expoente $(\gamma-1)/(n\gamma) \approx 0{,}1$), a
potência correspondente ao ponto da Tabela 13 cai apenas de ≈ 1108 kW para ≈ 1069 kW (anular
físico) ou ≈ 1049 kW (as três correções juntas, −5 %). O custo com gás combustível e os
custos de capital do compressor são superestimados nessa mesma proporção; as conclusões
qualitativas do artigo não mudam.

## Itens 4–6 · Correlações de custo (Tabela 18, cenário 1)

In [4]:
k = matlab["case2_design_220bar"]
d_in, lengths = np.array(k["xopt"][8:12]), [6020.0126, 6020.0126, 4520.0126, 4520.0126]
power = k["out_CompressorPower"]
q_gas_total = sum(k["q_gas"])
pg = k["out_DischargePressure"]
from gaslift.economics import compressor_power
rows = {
    "code (paper numbers)": {"Compressor power (kW)": power, "CC (10⁶ USD)": compressor_capital_cost(power, "hp"), "CT (10⁶ USD)": turbine_capital_cost(power, "hp"), "CP (10⁶ USD)": pipeline_capital_cost(d_in, lengths, "matlab")},
    "4: McCoy with D^a7": {"Compressor power (kW)": power, "CC (10⁶ USD)": compressor_capital_cost(power, "hp"), "CT (10⁶ USD)": turbine_capital_cost(power, "hp"), "CP (10⁶ USD)": pipeline_capital_cost(d_in, lengths, "mccoy")},
    "5: γ = 1.25": {"Compressor power (kW)": compressor_power(pg, q_gas_total, CompressorParameters(polytropic_exponent=1.25)), "CC (10⁶ USD)": np.nan, "CT (10⁶ USD)": np.nan, "CP (10⁶ USD)": np.nan},
    "6: Couper correlations in kW": {"Compressor power (kW)": power, "CC (10⁶ USD)": compressor_capital_cost(power, "kW"), "CT (10⁶ USD)": turbine_capital_cost(power, "kW"), "CP (10⁶ USD)": pipeline_capital_cost(d_in, lengths, "matlab")},
}
t = pd.DataFrame(rows).T
t["ISBL (10⁶ USD)"] = t[["CC (10⁶ USD)", "CT (10⁶ USD)", "CP (10⁶ USD)"]].sum(axis=1, min_count=3)
t

,Compressor power (kW),CC (10⁶ USD),CT (10⁶ USD),CP (10⁶ USD),ISBL (10⁶ USD)
code (paper numbers),1797.4517,1.7874,3.9466,1.0524,6.7865
4: McCoy with D^a7,1797.4517,1.7874,3.9466,1.1197,6.8538
5: γ = 1.25,1696.3617,NaN,NaN,NaN,NaN
6: Couper correlations in kW,1797.4517,1.4901,3.1117,1.0524,5.6542


## Item 7 · Geometria impressa nas Tabelas 2 e 7

Simulando o Caso 1 com a geometria impressa no artigo (coluna de produção de 5 in, anular de
1500 m, trecho inferior de 300 m, rugosidade de 100 µm nas linhas de gás) os valores da
Fig. 6 mudam substancialmente — o que confirma que os números publicados vieram do
`case_03.m` e não da Tabela 2.

In [5]:
points = [(10.0, 2.0), (5.0, 1.0), (15.0, 3.0)]
rows = {}
for qo, qg in points:
    a = case1_single_well(q_oil=qo, q_gas=qg).simulator(sim_opts).solve()
    b = case1_single_well(q_oil=qo, q_gas=qg, wellbore_diameter_in=5.0, annulus_length=1500.0, wellbore_length=1800.0, gas_line_roughness=100e-6).simulator(sim_opts).solve()
    rows[f"oil {qo:g}, gas {qg:g} kg/s"] = {"Downhole, code geometry (bar)": a.pressure_bar("well_1:downhole"), "Downhole, Table 2 geometry (bar)": b.pressure_bar("well_1:downhole"),
                                             "Node 9, code geometry (bar)": a.pressure_bar("well_1:gas_junction"), "Node 9, Table 2 geometry (bar)": b.pressure_bar("well_1:gas_junction")}
pd.DataFrame(rows).T

,"Downhole, code geometry (bar)","Downhole, Table 2 geometry (bar)","Node 9, code geometry (bar)","Node 9, Table 2 geometry (bar)"
"oil 10, gas 2 kg/s",213.8629,178.1522,295.9956,334.6503
"oil 5, gas 1 kg/s",179.5794,160.0739,201.5887,203.8037
"oil 15, gas 3 kg/s",260.5147,205.6426,408.5073,491.2773


## Contagem de equações

Para o Caso 2 o artigo informa 232 equações; a formulação da Seção 3.3 produz
$3N + 4S = 3 \cdot 28 + 4 \cdot 27 = 192$ (o Caso 1 tem $3 \cdot 10 + 4 \cdot 9 = 66$, como publicado).

In [6]:
for case in (case1_single_well(), case2_multi_well()):
    lay = case.network.layout
    print(f"{case.network.name}: N = {lay.n_nodes}, S = {lay.n_pipes}, 3N + 4S = {lay.size}, specifications N + S = {lay.n_specifications}")

Case 1 - offshore single well: N = 10, S = 9, 3N + 4S = 66, specifications N + S = 19
Case 2 - offshore multiple wells: N = 28, S = 27, 3N + 4S = 192, specifications N + S = 55
